# FinTech Credit & Loan Review: Relational Model + Quality Reports

Builds the five-table DuckDB model from `data/structured/raw/` and exports the curated quality reports to `reports/`. The logic lives in `src/database/build_relational_model.py` and `src/database/generate_quality_reports.py`; this notebook runs them and shows the results.

**Local:** open this notebook from the repository and run all cells.
**Colab:** run the first cell to clone the repository.

In [ ]:
import os, sys, subprocess

if 'google.colab' in sys.modules and not os.path.exists('FinTech-Credit-Loan-Review'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Wenting-Hou/FinTech-Credit-Loan-Review.git'], check=True)

# Work from the repository root
for root in ['FinTech-Credit-Loan-Review', '..', '.']:
    if os.path.exists(os.path.join(root, 'data', 'structured', 'raw')):
        os.chdir(root)
        break
print(os.getcwd())

## Step 1: Build the relational model

Five tables with primary and foreign keys: `applicants`, `loan_products`, `loan_applications`, `credit_pull_events`, `applicant_risk_features`. Money and dates are normalized on load; applications dated after the 2026-09-14 extract date are flagged, not dropped.

In [ ]:
def run(script):
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    print(result.stdout, result.stderr)
    result.check_returncode()

run('src/database/build_relational_model.py')

## Step 2: Quality reports (the curated structured outputs)

In [ ]:
run('src/database/generate_quality_reports.py')

## Step 3: Look at the results

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect('fintech.duckdb', read_only=True)
con.sql('SHOW TABLES').df()

In [ ]:
pd.read_csv('reports/quarantine_summary.csv')

In [ ]:
# Repeat applicants: one person's applications side by side
pd.read_csv('reports/repeat_applicants.csv').head(10)

In [ ]:
# Applications per product and preliminary decision (quarantined rows excluded)
con.sql('''
    SELECT product_code, preliminary_decision, COUNT(*) AS applications
    FROM loan_applications
    WHERE NOT after_extract_date
    GROUP BY ALL ORDER BY product_code, preliminary_decision
''').df()

In [ ]:
# Release the database so Step 1 can be run again
con.close()